In [32]:
import polars as pl

# download inphared metadata
inphared_meta = pl.read_csv('https://millardlab-inphared.s3.climb.ac.uk/14Apr2025_data.tsv.gz', separator='\t')
print("Number of reference phages:", inphared_meta.shape[0])

Number of reference phages: 34076


In [33]:
# download ehomd metadata
ehomd_genera = set(
    pl.read_csv('https://www.ehomd.org/download/dld_static_file/GTDB_taxonomy20250406.csv', separator='\t', skip_rows=1)
        .filter(
            (pl.col('GTDB Taxonomy').str.contains(';g_')) &
            (pl.col('GTDB Taxonomy').str.contains(';s_'))
        )
        .with_columns(pl.col('GTDB Taxonomy').str.split(';g__').list[1].str.split(';s_').list[0].alias('gtdb_genus'))
        ['gtdb_genus']
)
print("Number of oral genera:", len(ehomd_genera))

Number of oral genera: 237


In [36]:
# identify reference phages with hosts in oral genera
inphared_oral = (
    inphared_meta
        .with_columns([
            pl.col('Isolation Host (beware inconsistent and nonsense values)').str.split(' ').list[0].alias('host_genus')
        ])
        .filter(pl.col('host_genus').is_in(ehomd_genera))
)
print("Number of phages with oral host:", inphared_oral.shape[0])

Number of phages with oral host: 11293


In [37]:
# write out list of oral phages
inphared_oral[['Accession']].write_csv('inphared_oral_phages.txt')

In [38]:
# download reference phage sequences
!wget https://millardlab-inphared.s3.climb.ac.uk/14Apr2025_genomes.fa.gz

--2025-05-28 15:11:52--  https://millardlab-inphared.s3.climb.ac.uk/14Apr2025_genomes.fa.gz
Resolving millardlab-inphared.s3.climb.ac.uk (millardlab-inphared.s3.climb.ac.uk)... 147.188.173.14
Connecting to millardlab-inphared.s3.climb.ac.uk (millardlab-inphared.s3.climb.ac.uk)|147.188.173.14|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 643340443 (614M) [binary/octet-stream]
Saving to: ‘14Apr2025_genomes.fa.gz’

14Apr2025_genomes.f 100%[===================>] 613.54M  5.88MB/s    in 1m 44s  

2025-05-28 15:13:42 (5.90 MB/s) - ‘14Apr2025_genomes.fa.gz’ saved [643340443/643340443]



In [39]:
# extract oral phages from reference sequences
!seqkit grep \
    14Apr2025_genomes.fa.gz \
    --pattern-file inphared_oral_phages.txt \
    --threads 4 \
    --out-file 14Apr2025_genomes_oral_phages.fa.gz

[INFO] 11294 patterns loaded from file


In [ ]:
%%bash
# dereplicate oral phages with vClust
vclust prefilter \
    -i 14Apr2025_genomes_oral_phages.fa.gz \
    -o fltr.txt \
    --min-ident 0.95 \
    --threads 8

vclust align \
    -i 14Apr2025_genomes_oral_phages.fa.gz \
    -o ani.tsv \
    --filter fltr.txt \
    --threads 8

vclust cluster \
    -i ani.tsv \
    -o clusters.tsv \
    --ids ani.ids.tsv \
    --algorithm leiden \
    --metric ani \
    --ani 0.95 \
    --qcov 0.85 \
    --out-repr

In [3]:
# extract cluster representatives
!cut -f 2 -d$'\t' clusters.tsv | tail -n +2 > clusters.txt

!seqkit grep \
    14Apr2025_genomes_oral_phages.fa.gz \
    --pattern-file clusters.txt \
    --threads 4 \
    --out-file 14Apr2025_genomes_oral_votu_reps.fa.gz

[INFO] 3855 patterns loaded from file


In [4]:
# split oral phage reps into 5 files
!seqkit split \
    14Apr2025_genomes_oral_votu_reps.fa.gz \
    --by-part 5 \
    --out-dir oral_votu_reps_split \
    --threads 8

[INFO] split into 5 parts
[INFO] read sequences ...
[INFO] read 3855 sequences
[INFO] write 771 sequences to file: oral_votu_reps_split/14Apr2025_genomes_oral_votu_reps.part_001.fa.gz
[INFO] write 771 sequences to file: oral_votu_reps_split/14Apr2025_genomes_oral_votu_reps.part_002.fa.gz
[INFO] write 771 sequences to file: oral_votu_reps_split/14Apr2025_genomes_oral_votu_reps.part_003.fa.gz
[INFO] write 771 sequences to file: oral_votu_reps_split/14Apr2025_genomes_oral_votu_reps.part_004.fa.gz
[INFO] write 771 sequences to file: oral_votu_reps_split/14Apr2025_genomes_oral_votu_reps.part_005.fa.gz
